# 04 — AI-Accelerated Workflow (drivers commentary)

A programmatic LLM component that writes the daily desk note **only from our computed metrics**, with a number-grounding guard.

- LLM called from code (Anthropic API), key via env var only.
- Prompts, outputs, and failure modes logged to `outputs/ai_logs/commentary_log.jsonl`.
- A guard flags any number in the output that is not grounded in the metrics.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import ai_commentary as ai
import json

metrics = ai.collect_metrics()
print(json.dumps(metrics, indent=2))

{
  "market": "DE-LU",
  "delivery_period": "prompt month (30d) from 2026-05-16",
  "fair_value_baseload_p50": 86.47,
  "fair_value_baseload_p10": 83.38,
  "fair_value_baseload_p90": 89.69,
  "fair_value_peak": 55.39,
  "market_anchor_baseload": 84.44,
  "level_edge_eur_mwh": 2.03,
  "level_band_halfwidth": 3.16,
  "level_direction": "FLAT",
  "level_position_mw": 0.0,
  "shape_spread_forecast": -22.26,
  "shape_spread_anchor": -20.19,
  "shape_direction": "SHORT peak/base spread",
  "model_cv_mae": {
    "hgb": 16.27,
    "linear": 19.44,
    "seasonal_naive": 35.36
  },
  "invalidation": [
    {
      "name": "Realised avg within P10-P90 band",
      "pass": false,
      "detail": "actual 94.47 vs [83.38, 89.69]"
    },
    {
      "name": "Renewable forecast drift <= 15%",
      "pass": true,
      "detail": "realised vs forecast renewables drift +0.7%"
    }
  ]
}


## Generate the note
Uses the Anthropic API if `ANTHROPIC_API_KEY` is set; otherwise logs a `no_api_key` failure mode and uses the deterministic fallback (so the pipeline always produces a note).

In [2]:
text, source, log = ai.generate(metrics)
print(f"source = {source} | status = {log['status']}\n")
print(text)

source = fallback | status = no_api_key

DE-LU prompt month (30d) from 2026-05-16. Fair value (baseload P50) 86.47 EUR/MWh vs market anchor 84.44, an edge of 2.03 against a band half-width of 3.16. Level signal: FLAT (position 0.0 MW). Shape: SHORT peak/base spread (forecast peak-base -22.26 vs anchor -20.19). Model skill: HGB CV MAE 16.27 vs seasonal-naive 35.36. Invalidation: Realised avg within P10-P90 band FAIL; Renewable forecast drift <= 15% PASS.


## The hallucination guard (failure mode)
We simulate an LLM output that invents numbers (gas 42.0, price 150.0) to show the guard catches them. Real outputs that fail this check are rejected in favour of the safe fallback.

In [3]:
good = f"Fair value {metrics['fair_value_baseload_p50']} vs anchor {metrics['market_anchor_baseload']}."
bad = good + " Gas at 42.0 could push price to 150.0."
print("grounded output  -> ungrounded numbers:", ai.verify_numbers(good, metrics))
print("hallucinated out -> ungrounded numbers:", ai.verify_numbers(bad, metrics))

grounded output  -> ungrounded numbers: []
hallucinated out -> ungrounded numbers: [42.0, 150.0]


## The audit log
Every call logs prompt + output + status (no secrets).

In [4]:
for line in open(ai.LOG_PATH):
    rec = json.loads(line)
    print(rec["timestamp_utc"], "|", rec["status"], "|", rec["model"])

2026-06-16T00:08:40.630538+00:00 | no_api_key | claude-haiku-4-5-20251001
2026-06-16T00:09:57.363537+00:00 | no_api_key | claude-haiku-4-5-20251001
